# Yes/No ASR Tutorial

### ဟီးဘရူး (Hebrew) ဘာသာနဲ့ အသံသွင်းထားတဲ့ "Yes", "No" ဒေတာကိုသုံးပြီး Automatic Speech Recognition (ASR) လုပ်ကြည့်ကြရအောင်။    

#### For AI (Fundamental) Engineering Class
#### By Dr. Ye Kyaw Thu, Lab Leader, Language Understanding Lab, AI Mario, Myanmar
#### Last Updated: 25 Aug 2026  


ကိုယ့်စက်ထဲမှာ [Kaldi](https://github.com/kaldi-asr/kaldi) Framework ကတော့  installation လုပ်ထားပြီးသား အခြေအနေ ဖြစ်ရပါမယ်။   
Installation မှာ တက်တတ်တဲ့ error အများစုကတော့ python library မရှိတာကြောင့်၊ gcc, g++ version မတူတာကြောင့်နဲ့ GPU ကို detect မဖြစ်တာကြောင့်ပါ။   

ဆရာက ဘာကြောင့် Kaldi Framework ကိုသုံးပြီး သင်ပေးရတာလဲလို့ မေးရင် အဖြေကတော့ Speech processing, ASR, TTS အလုပ်တွေကို ထဲထဲဝင်ဝင် တတ်စေချင်လို့ပါ။   

Kaldi: [https://github.com/kaldi-asr/kaldi](https://github.com/kaldi-asr/kaldi)  
Kaldi Document: [https://kaldi-asr.org/doc/index.html](https://kaldi-asr.org/doc/index.html)    

In [1]:
%pwd

'/mnt/disk1/ye/exp/speech/kaldi/egs/yesno_demo'

In [2]:
!ls ../

aidatatang_200zh      demo		       madcat_zh	 sre16
aishell		      dihard_2018	       malach		 svhn
aishell2	      fame		       mandarin_bn_bc	 swahili
ami		      farsdat		       material		 swbd
an4		      fisher_callhome_spanish  mgb2_arabic	 tedlium
apiai_decode	      fisher_english	       mgb5		 thchs30
aspire		      fisher_swbd	       mini_librispeech  tidigits
aurora4		      formosa		       mobvoi		 timit
babel		      gale_arabic	       mobvoihotwords	 tunisian_msa
babel_multilang       gale_mandarin	       multi_cn		 uw3
bentham		      gigaspeech	       multi_en		 voxceleb
bn_music_speech       gop_speechocean762       nsc		 voxforge
callhome_diarization  gp		       opensat20	 vystadial_cz
callhome_egyptian     heroico		       ptb		 vystadial_en
casia_hwdb	      hi_mia		       README.txt	 wenetspeech
chime1		      hkust		       reverb		 wsj
chime2		      hub4_english	       rimes		 xbmu_amdo31
chime3		      hub4_spanish	       rm		 yesno
chime4		      iam		       sad_rats		 yesno

In [3]:
!ls 

ASR_Kaldi_Tutorial_No1.ipynb


yesno ဖိုလ်ဒါအောက်ထဲက ရှိတဲ့ shell script တွေကို ယူသုံးမှာမို့ လက်ရှိ ဒီမိုလုပ်ပြမယ့် ဖိုလ်ဒါအောက်ထဲကို yesno/ ဖိုလ်ဒါနဲ့ အဲဒီအောက်မှာ ရှိတဲ့ ဖိုင်/ဖိုလ်ဒါ အကုန်ကို ကော်ပီကူးယူလိုက်မယ်။  

In [4]:
!cp -r ../yesno/* .

In [5]:
!ls

ASR_Kaldi_Tutorial_No1.ipynb  README.txt  s5


In [43]:
!cat /mnt/disk1/ye/exp/speech/kaldi/egs/yesno_demo/README.txt



The "yesno" corpus is a very small dataset of recordings of one individual
saying yes or no multiple times per recording, in Hebrew.  It is available from
http://www.openslr.org/1.
It is mainly included here as an easy way to test out the Kaldi scripts.

The test set is perfectly recognized at the monophone stage, so the dataset is
not exactly challenging.

The scripts are in s5/.



In [6]:
%pwd

'/mnt/disk1/ye/exp/speech/kaldi/egs/yesno_demo'

In [7]:
!tree .

.
├── ASR_Kaldi_Tutorial_No1.ipynb
├── README.txt
└── s5
    ├── conf
    │   ├── mfcc.conf
    │   └── topo_orig.proto
    ├── input
    │   ├── lexicon_nosil.txt
    │   ├── lexicon.txt
    │   ├── phones.txt
    │   └── task.arpabo
    ├── local
    │   ├── create_yesno_txt.pl
    │   ├── create_yesno_waves_test_train.pl
    │   ├── create_yesno_wav_scp.pl
    │   ├── prepare_data.sh
    │   ├── prepare_dict.sh
    │   ├── prepare_lm.sh
    │   └── score.sh -> ../steps/score_kaldi.sh
    ├── path.sh
    ├── run.sh
    ├── steps -> ../../wsj/s5/steps
    └── utils -> ../../wsj/s5/utils

7 directories, 17 files


## Let's learn run.sh

```bash
#!/usr/bin/env bash

train_cmd="utils/run.pl"
decode_cmd="utils/run.pl"

if [ ! -d waves_yesno ]; then
  wget http://www.openslr.org/resources/1/waves_yesno.tar.gz || exit 1;
  # was:
  # wget http://sourceforge.net/projects/kaldi/files/waves_yesno.tar.gz || exit 1;
  tar -xvzf waves_yesno.tar.gz || exit 1;
fi

train_yesno=train_yesno
test_base_name=test_yesno

rm -rf data exp mfcc

# Data preparation

local/prepare_data.sh waves_yesno
local/prepare_dict.sh
utils/prepare_lang.sh --position-dependent-phones false data/local/dict "<SIL>" data/local/lang data/lang
local/prepare_lm.sh

# Feature extraction
for x in train_yesno test_yesno; do
 steps/make_mfcc.sh --nj 1 data/$x exp/make_mfcc/$x mfcc
 steps/compute_cmvn_stats.sh data/$x exp/make_mfcc/$x mfcc
 utils/fix_data_dir.sh data/$x
done

# Mono training
steps/train_mono.sh --nj 1 --cmd "$train_cmd" \
  --totgauss 400 \
  data/train_yesno data/lang exp/mono0a

# Graph compilation
utils/mkgraph.sh data/lang_test_tg exp/mono0a exp/mono0a/graph_tgpr

# Decoding
steps/decode.sh --nj 1 --cmd "$decode_cmd" \
    exp/mono0a/graph_tgpr data/test_yesno exp/mono0a/decode_test_yesno

for x in exp/*/decode*; do [ -d $x ] && grep WER $x/wer_* | utils/best_wer.sh; done
```

## path.sh

In [8]:
%cd s5

/mnt/disk1/ye/exp/speech/kaldi/egs/yesno_demo/s5


In [9]:
!cat path.sh

export KALDI_ROOT=`pwd`/../../..
[ -f $KALDI_ROOT/tools/env.sh ] && . $KALDI_ROOT/tools/env.sh
export PATH=$PWD/utils/:$KALDI_ROOT/tools/openfst/bin:$PWD:$PATH
[ ! -f $KALDI_ROOT/tools/config/common_path.sh ] && echo >&2 "The standard file $KALDI_ROOT/tools/config/common_path.sh is not present -> Exit!" && exit 1
. $KALDI_ROOT/tools/config/common_path.sh
export LC_ALL=C
export PYTHONUNBUFFERED=1




%env ရှိပေမဲ့ path.sh ထဲမှာ ရေးထားသလိုမျိုးကို Jupyter notebook cell ထဲမှာ setup လုပ်ဖို့က အရမ်းအဆင်ပြေကြီးမဟုတ်လို့ Python ရဲ့ environment variable အနေနဲ့ပဲ setup လုပ်ကြည့်မယ်။  
ကိုယ့်စက်ထဲမှာ run တဲ့အခါမှာတော့ path တွေကို ပြန်ပြင်ရလိမ့်မယ်။  

## Setup Python Environment Variables

In [14]:
import os

# 1. Define the root of your Kaldi installation
kaldi_root = "/mnt/disk1/ye/exp/speech/kaldi"

# 2. Define the path to your recipe
recipe_path = os.path.join(kaldi_root, "egs/yesno_demo/s5")

# 3. Set KALDI_ROOT
os.environ["KALDI_ROOT"] = kaldi_root

# 4. Add all necessary Kaldi binary paths to the system PATH
# This mimics exactly what path.sh does!
paths_to_add = [
    os.path.join(recipe_path, "utils"),
    os.path.join(recipe_path, "steps"),
    recipe_path,
    os.path.join(kaldi_root, "tools/openfst/bin"),
    os.path.join(kaldi_root, "src/featbin"),
    os.path.join(kaldi_root, "src/gmmbin"),
    os.path.join(kaldi_root, "src/bin"),
    os.path.join(kaldi_root, "src/fstbin"),
    os.path.join(kaldi_root, "src/lmbin"),
    os.path.join(kaldi_root, "src/decoderbin"),
    os.path.join(kaldi_root, "src/nnet3bin"),
    os.path.join(kaldi_root, "src/ivectorbin"),
    os.path.join(kaldi_root, "src/online2bin")
]

# Get the current PATH and append our new paths
current_path = os.environ.get("PATH", "")
os.environ["PATH"] = ":".join(paths_to_add) + ":" + current_path

# 5. Verify it worked
print("KALDI_ROOT:", os.environ.get("KALDI_ROOT"))
print("PATH includes kaldi utils:", "egs/yesno_demo/s5/utils" in os.environ.get("PATH", ""))

KALDI_ROOT: /mnt/disk1/ye/exp/speech/kaldi
PATH includes kaldi utils: True


## Step 0: Clean up previous runs and Download the Data

In [15]:
%%bash
cd /mnt/disk1/ye/exp/speech/kaldi/egs/yesno_demo/s5

# Download data
rm -rf data exp mfcc waves_yesno
wget http://www.openslr.org/resources/1/waves_yesno.tar.gz
tar -xvzf waves_yesno.tar.gz

--2026-08-25 21:02:41--  http://www.openslr.org/resources/1/waves_yesno.tar.gz
Resolving www.openslr.org (www.openslr.org)... 136.243.171.4
Connecting to www.openslr.org (www.openslr.org)|136.243.171.4|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4703754 (4.5M) [application/x-gzip]
Saving to: ‘waves_yesno.tar.gz’

     0K .......... .......... .......... .......... ..........  1% 35.4K 2m8s
    50K .......... .......... .......... .......... ..........  2% 20.5K 2m53s
   100K .......... .......... .......... .......... ..........  3% 26.7K 2m49s
   150K .......... .......... .......... .......... ..........  4% 18.4K 3m5s
   200K .......... .......... .......... .......... ..........  5% 25.4K 3m1s
   250K .......... .......... .......... .......... ..........  6% 24.2K 2m58s
   300K .......... .......... .......... .......... ..........  7% 26.8K 2m54s
   350K .......... .......... .......... .......... ..........  8% 31.8K 2m47s
   400K .......... ........

waves_yesno/
waves_yesno/1_0_0_0_0_0_1_1.wav
waves_yesno/1_1_0_0_1_0_1_0.wav
waves_yesno/1_0_1_1_1_1_0_1.wav
waves_yesno/1_1_1_1_0_1_0_0.wav
waves_yesno/0_0_1_1_1_0_0_0.wav
waves_yesno/0_1_1_1_1_1_1_1.wav
waves_yesno/0_1_0_1_1_1_0_0.wav
waves_yesno/1_0_1_1_1_0_1_0.wav
waves_yesno/1_0_0_1_0_1_1_1.wav
waves_yesno/0_0_1_0_1_0_0_0.wav
waves_yesno/0_1_0_1_1_0_1_0.wav
waves_yesno/0_0_1_1_0_1_1_0.wav
waves_yesno/1_0_0_0_1_0_0_1.wav
waves_yesno/1_1_0_1_1_1_1_0.wav
waves_yesno/0_0_1_1_1_1_0_0.wav
waves_yesno/1_1_0_0_1_1_1_0.wav
waves_yesno/0_0_1_1_0_1_1_1.wav
waves_yesno/1_1_0_1_0_1_1_0.wav
waves_yesno/0_1_0_0_0_1_1_0.wav
waves_yesno/0_0_0_1_0_0_0_1.wav
waves_yesno/0_0_1_0_1_0_1_1.wav
waves_yesno/0_0_1_0_0_0_1_0.wav
waves_yesno/1_1_0_1_1_0_0_1.wav
waves_yesno/0_1_1_1_0_1_0_1.wav
waves_yesno/0_1_1_1_0_0_0_0.wav
waves_yesno/README~
waves_yesno/0_1_0_0_0_1_0_0.wav
waves_yesno/1_0_0_0_0_0_0_1.wav
waves_yesno/1_1_0_1_1_0_1_1.wav
waves_yesno/1_1_0_0_0_0_0_1.wav
waves_yesno/1_0_0_0_0_0_0_0.wav
waves_y

In [38]:
from IPython.display import Audio, display
display(Audio("waves_yesno/0_0_0_0_1_1_1_1.wav"))

In [39]:
from IPython.display import Audio, display
display(Audio("waves_yesno/0_0_0_1_0_1_1_0.wav"))

## Verify

In [15]:
!ls waves_yesno | head -n 5

0_0_0_0_1_1_1_1.wav
0_0_0_1_0_0_0_1.wav
0_0_0_1_0_1_1_0.wav
0_0_1_0_0_0_1_0.wav
0_0_1_0_0_1_1_0.wav


In [16]:
!ls waves_yesno/*.wav | wc

     60      60    1920


## Step 1: Data Preparation (Mapping Audio to Text)

This step creates the standard Kaldi data files (wav.scp, text, utt2spk, spk2utt) in the data/ folder.

In [17]:
%%bash
cd /mnt/disk1/ye/exp/speech/kaldi/egs/yesno_demo/s5

# Run data prep
local/prepare_data.sh waves_yesno

Preparing train and test data


Let's look at what Kaldi requires for training.

In [19]:
%%bash

# Check the text file (Utterance ID mapped to transcript)
echo "=== data/train_yesno/text ==="
head -n 3 data/train_yesno/text

=== data/train_yesno/text ===
0_0_0_0_1_1_1_1 NO NO NO NO YES YES YES YES
0_0_0_1_0_0_0_1 NO NO NO YES NO NO NO YES
0_0_0_1_0_1_1_0 NO NO NO YES NO YES YES NO


Kaldi က raw audio ဖိုင်ကို ဖတ်တာမဟုတ်ဘူး။ တကယ်တမ်း ဖတ်မှာက wav.scp ဆိုတဲ့ ဖိုင်ပါ။ wav.scp ဆိုတဲ့ ဖိုင်ထဲမှာက အောက်မှာ မြင်ရတဲ့အတိုင်းပါပဲ၊ audio ဖိုင်နဲ့ သက်ဆိုင်ရာ ground-truth transcription အတွဲတွေပါ။  

In [20]:
%%bash

# Check the wav.scp file (Utterance ID mapped to audio file path)
echo "=== data/train_yesno/wav.scp ==="
head -n 3 data/train_yesno/wav.scp

=== data/train_yesno/wav.scp ===
0_0_0_0_1_1_1_1 waves_yesno/0_0_0_0_1_1_1_1.wav
0_0_0_1_0_0_0_1 waves_yesno/0_0_0_1_0_0_0_1.wav
0_0_0_1_0_1_1_0 waves_yesno/0_0_0_1_0_1_1_0.wav


## Step 2: Dictionary Preparation (Words to Phonemes)  

ဒီအဆင့်မှာ pronunciation dictionary (အသံထွက် အဘိဓာန်) ကို ဆောက်မှာပါ။ ဒီတစ်ခေါက် လုပ်မယ့် ASR က `yes/no` မို့လို့ mapping က ရိုးရှင်းပါတယ်။ "YES" အတွက် phoneme က "Y EH S" ဖြစ်ပြီး၊ "NO" အတွက်ကတော့ "N OW" ပါ။

In [21]:
%%bash

local/prepare_dict.sh

Dictionary preparation succeeded


ဆောက်ပြီးသွားတဲ့ Pronunciation dictionary ဖိုင်ကို စစ်ဆေးကြည့်ရအောင်။  


In [22]:
%%bash

echo "=== data/local/dict/lexicon.txt ==="
cat data/local/dict/lexicon.txt

=== data/local/dict/lexicon.txt ===
<SIL> SIL
YES Y
NO N


## Step 3: Language Model & FST Compilation

ဒီအဆင့်မှာတော့ Finite State Transducers (FSTs) ကို ဆောက်ပါမယ်။ သူက ဘာလုပ်ပေးတာလဲ ဆိုတော့ အထက်က ဆောက်ခဲ့တဲ့ အသံထွက်အဘိဓာန်နဲ့ language model (သဒ္ဒါ) ကို ပေါင်းလိုက်ပြီး Kaldi decoder က နားလည်တဲ့ graph format အဖြစ်ပြောင်းတဲ့ အပိုင်းပါ။ ဆရာ နောက်လာမယ့် စာသင်ချိန်မှာတော့ FSA, FST တွေအကြောင်းကို သင်ကြားပေးပါမယ်။

In [23]:
%%bash

# Build the L.fst (Lexicon) and other base FSTs
utils/prepare_lang.sh --position-dependent-phones false data/local/dict "<SIL>" data/local/lang data/lang

utils/prepare_lang.sh --position-dependent-phones false data/local/dict <SIL> data/local/lang data/lang
Checking data/local/dict/silence_phones.txt ...
--> reading data/local/dict/silence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/silence_phones.txt is OK

Checking data/local/dict/optional_silence.txt ...
--> reading data/local/dict/optional_silence.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/optional_silence.txt is OK

Checking data/local/dict/nonsilence_phones.txt ...
--> reading data/local/dict/nonsilence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/nonsilence_phones.txt is OK

Checking disjoint: silence_phones.txt, nonsilence_phones.txt
--> disjoint property is OK.

Checking data/local/dict/lexicon.txt
--> reading data/local/dict

fstaddselfloops data/lang/phones/wdisambig_phones.int data/lang/phones/wdisambig_words.int 


prepare_lang.sh: validating output directory
utils/validate_lang.pl data/lang
Checking existence of separator file
separator file data/lang/subword_separator.txt is empty or does not exist, deal in word case.
Checking data/lang/phones.txt ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/lang/phones.txt is OK

Checking words.txt: #0 ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/lang/words.txt is OK

Checking disjoint: silence.txt, nonsilence.txt, disambig.txt ...
--> silence.txt and nonsilence.txt are disjoint
--> silence.txt and disambig.txt are disjoint
--> disambig.txt and nonsilence.txt are disjoint
--> disjoint property is OK

Checking sumation: silence.txt, nonsilence.txt, disambig.txt ...
--> found no unexplainable phones in phones.txt

Checking data/lang/phones/context_indep.{txt, int, csl} ...
--> text seems to be UTF-8 or ASCII, checking whitespa

G.fst ကိုဆောက်မယ်။  

In [24]:
%%bash

# Build the G.fst (Grammar/Language Model) and create the final test folder
local/prepare_lm.sh

Preparing language models for test


arpa2fst --disambig-symbol=#0 --read-symbol-table=data/lang_test_tg/words.txt input/task.arpabo data/lang_test_tg/G.fst 
LOG (arpa2fst[5.5.1182~2-e02e3]:Read():arpa-file-parser.cc:94) Reading \data\ section.
LOG (arpa2fst[5.5.1182~2-e02e3]:Read():arpa-file-parser.cc:149) Reading \1-grams: section.
LOG (arpa2fst[5.5.1182~2-e02e3]:RemoveRedundantStates():arpa-lm-compiler.cc:359) Reduced num-states from 1 to 1
fstisstochastic data/lang_test_tg/G.fst 


1.20397 1.20397
Succeeded in formatting data.



Output အဖြစ် ရရှိလာတဲ့ language model ဖိုလ်ဒါအောက်မှာ ရှိတဲ့ ဖိုင်တွေကို လေ့လာကြည့်ကြရအောင်။  

In [25]:
%%bash

# Look at the generated language model folder
ls data/lang_test_tg/

G.fst
L_disambig.fst
L.fst
oov.int
oov.txt
phones
phones.txt
topo
words.txt


အထက်မှာ မြင်နေရတဲ့ ဖိုင်တွေထဲမှာ အရေးကြီးဆုံးက G.fst (သို့) Grammar FST ဖိုင်ပါပဲ။ အဲဒီဖိုင်ထဲမှာ တွဲလို့ ရနိုင်တဲ့ စာလုံးအတွဲတွေရဲ့ သဒ္ဒါကို သိမ်းထားတာမို့ပါ။ ဒီ experiment အသေးလေးမှာတော "YES ပြီးရင် NO" လာနိုင်တယ်ဆိုတဲ့ အစီအစဉ်မျိုး (i.e. probability of word sequences) ပေါ့။  

## Step 4: Feature Extraction (Audio to MFCCs)

ASR model ကို waveform ပုံစံကနေ ဆောက်လို့မရပါဘူး။ ဆရာ အမြဲပြောခဲ့သလိုပါပဲ feature ထုတ်တဲ့ကဏ္ဍ လာပါပြီ။ ဒီ experiment မှာတော့ Mel-Frequency Cepstral Coefficients (MFCCs) လို့ခေါ်တဲ့ feature တွေကိုဆွဲထုတ်ပြီး၊ အသံသွင်းစဉ်က မိုက်တွေမတူရင်လည်း အဆင်ပြေအောင်လို့ ရလာတဲ့ feature တွေကို Cepstral Mean Normalization (CMVN) လုပ်ပါမယ်။ အဲဒါကို training အတွက်ရော၊ testing အတွက်ရော လုပ်ရပါမယ်။  သုံးထားတဲ့ variable နာမည်တွေနဲ့ ပြောရရင် `train_yesno` နဲ့ `test_yesno` နှစ်မျိုးစလုံးအတွက်ပါ။  


In [27]:
%%bash

# 4.1 Extract MFCCs for Training data
steps/make_mfcc.sh --nj 1 data/train_yesno exp/make_mfcc/train_yesno mfcc
steps/compute_cmvn_stats.sh data/train_yesno exp/make_mfcc/train_yesno mfcc
utils/fix_data_dir.sh data/train_yesno

steps/make_mfcc.sh --nj 1 data/train_yesno exp/make_mfcc/train_yesno mfcc
utils/validate_data_dir.sh: WARNING: you have only one speaker.  This probably a bad idea.
   Search for the word 'bold' in http://kaldi-asr.org/doc/data_prep.html
   for more information.
utils/validate_data_dir.sh: Successfully validated data-directory data/train_yesno
steps/make_mfcc.sh: [info]: no segments file exists: assuming wav.scp indexed by utterance.
steps/make_mfcc.sh: Succeeded creating MFCC features for train_yesno
steps/compute_cmvn_stats.sh data/train_yesno exp/make_mfcc/train_yesno mfcc
Succeeded creating CMVN stats for train_yesno
fix_data_dir.sh: kept all 31 utterances.
fix_data_dir.sh: old files are kept in data/train_yesno/.backup


In [28]:
%%bash

# 4.2 Extract MFCCs for Testing data
steps/make_mfcc.sh --nj 1 data/test_yesno exp/make_mfcc/test_yesno mfcc
steps/compute_cmvn_stats.sh data/test_yesno exp/make_mfcc/test_yesno mfcc
utils/fix_data_dir.sh data/test_yesno

steps/make_mfcc.sh --nj 1 data/test_yesno exp/make_mfcc/test_yesno mfcc
utils/validate_data_dir.sh: WARNING: you have only one speaker.  This probably a bad idea.
   Search for the word 'bold' in http://kaldi-asr.org/doc/data_prep.html
   for more information.
utils/validate_data_dir.sh: Successfully validated data-directory data/test_yesno
steps/make_mfcc.sh: [info]: no segments file exists: assuming wav.scp indexed by utterance.
steps/make_mfcc.sh: It seems not all of the feature files were successfully procesed (29 != 31); consider using utils/fix_data_dir.sh data/test_yesno
steps/make_mfcc.sh: Less than 95% the features were successfully generated. Probably a serious error.
steps/compute_cmvn_stats.sh data/test_yesno exp/make_mfcc/test_yesno mfcc
Succeeded creating CMVN stats for test_yesno
fix_data_dir.sh: kept 29 utterances out of 31
fix_data_dir.sh: old files are kept in data/test_yesno/.backup


mfcc ဖိုလ်ဒါကို ဝင်ကြည့်ရအောင်။  

In [29]:
%%bash

# Look at the extracted feature files
ls mfcc/

cmvn_test_yesno.ark
cmvn_test_yesno.scp
cmvn_train_yesno.ark
cmvn_train_yesno.scp
raw_mfcc_test_yesno.1.ark
raw_mfcc_test_yesno.1.scp
raw_mfcc_train_yesno.1.ark
raw_mfcc_train_yesno.1.scp


Here, the `.scp` files here point to the binary `.ark` files which contain the 13-dimensional MFCC float arrays for every 10ms of audio.  

## Step 5: Acoustic Model Training (Monophone GMM-HMM)


Acoustic model ဆောက်တဲ့ အပိုင်းပါ။  
"YES/NO" လို အရမ်းလွယ်တဲ့ ဒေတာမို့လို့ ဆောက်မယ့် မော်ဒယ်အမျိုးအစားက "Monophone GMM-HMM" လို့ခေါ်တဲ့ မော်ဒယ်ပါ။ သုံးသွားတာကို အသေးစိတ်ပြောရရင်တော့ အများဆုံး 400 Gaussians ထားမယ်ဆိုတဲ့ setting နဲ့ပါ။  

In [30]:
%%bash

time steps/train_mono.sh --nj 1 --cmd "utils/run.pl" --totgauss 400 data/train_yesno data/lang exp/mono0a

steps/train_mono.sh --nj 1 --cmd utils/run.pl --totgauss 400 data/train_yesno data/lang exp/mono0a
steps/train_mono.sh: Initializing monophone system.
steps/train_mono.sh: Compiling training graphs
steps/train_mono.sh: Aligning data equally (pass 0)
steps/train_mono.sh: Pass 1
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 2
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 3
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 4
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 5
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 6
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 7
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 8
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 9
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 10
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 11
steps/train_mono.sh: Pass 12
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 


real	0m9.431s
user	0m7.128s
sys	0m3.706s


In [31]:
# Check the final model file
!ls exp/mono0a/final.mdl

exp/mono0a/final.mdl


final.mdl ဆိုတဲ့ဖိုင်က acoustic model ပါ။ သူက ဘာလုပ်ပေးနိုင်လဲ ဆိုတော့ input လုပ်ပေးတဲ့ MFCC feature တွေကိုစစ်ဆေးပြီး HMM state probability တွေအဖြစ် output ထုတ်ပေးနိုင်ပါတယ်။ ဆိုလိုတာက ဖြစ်နိုင်ချေရှိတဲ့ phoneme တွေအဖြစ် mapping လုပ်ပေးတဲ့ အလုပ်ပါ။  

## Step 6: Decoding Graph Compilation

ဆရာတို့ test ဒေတာကိုသုံးပြီး decode မလုပ်ခင်မှာ အရေးကြီးတဲ့ အဆင့်တစ်ခု လုပ်ရပါသေးတယ်။ အဲဒါကတော့ HCLG.fst လို့ခေါ်တဲ့ ဖိုင်ဆောက်တဲ့ အပိုင်းပါ။ အဲဒီ transducer ဖိုင်နာမည်မှာ ဖော်ပြထားတဲ့အတိုင်းပါပဲ Acoustic Model (H) ရယ်၊ Context (C) ရယ်၊ Lexicon (L) ရယ်၊ ပြီးတော့ Language Model (G) ရယ်ကို အားလုံးတွဲထားတဲ့ graph ဖိုင်အကြီး တစ်ဖိုင်ထုတ်တဲ့ အဆင့်ပါ။  

In [32]:
%%bash

time utils/mkgraph.sh data/lang_test_tg exp/mono0a exp/mono0a/graph_tgpr

tree-info exp/mono0a/tree 
tree-info exp/mono0a/tree 
fstdeterminizestar --use-log=true 
fsttablecompose data/lang_test_tg/L_disambig.fst data/lang_test_tg/G.fst 
fstminimizeencoded 
fstpushspecial 
fstisstochastic data/lang_test_tg/tmp/LG.fst 


0.534295 0.533859
[info]: LG not stochastic.


fstcomposecontext --context-size=1 --central-position=0 --read-disambig-syms=data/lang_test_tg/phones/disambig.int --write-disambig-syms=data/lang_test_tg/tmp/disambig_ilabels_1_0.int data/lang_test_tg/tmp/ilabels_1_0.729473 data/lang_test_tg/tmp/LG.fst 
fstisstochastic data/lang_test_tg/tmp/CLG_1_0.fst 


0.534295 0.533859
[info]: CLG not stochastic.


make-h-transducer --disambig-syms-out=exp/mono0a/graph_tgpr/disambig_tid.int --transition-scale=1.0 data/lang_test_tg/tmp/ilabels_1_0 exp/mono0a/tree exp/mono0a/final.mdl 
fstminimizeencoded 
fstdeterminizestar --use-log=true 
fsttablecompose exp/mono0a/graph_tgpr/Ha.fst data/lang_test_tg/tmp/CLG_1_0.fst 
fstrmsymbols exp/mono0a/graph_tgpr/disambig_tid.int 
fstrmepslocal 
fstisstochastic exp/mono0a/graph_tgpr/HCLGa.fst 


0.5342 -0.000422432
HCLGa is not stochastic


add-self-loops --self-loop-scale=0.1 --reorder=true exp/mono0a/final.mdl exp/mono0a/graph_tgpr/HCLGa.fst 

real	0m0.087s
user	0m0.049s
sys	0m0.057s


In [33]:
# Check if HCLG.fst was created
!ls exp/mono0a/graph_tgpr/HCLG.fst

exp/mono0a/graph_tgpr/HCLG.fst


Decoder က ဝင်လာတဲ့ audio feature တွေအပေါ်မှာ မူတည်ပြီးတော့ ဒီအထက်က HCLG.fst ဂရဖ်ဖိုင်ထဲကနေ အဖြစ်နိုင်ဆုံး စာလုံးတွဲ (i.e. word sequence) ကိုရှာဖွေတာ ဖြစ်ပါတယ်။  

## Step 7: Decoding (Testing the Model)


ဒီတခါတော့ decoder ကို run ကြည့်ကြရအောင်။   
(It reads the test audio features and uses HCLG.fst to generate a lattice of possible transcriptions.)  

In [34]:
%%bash

time steps/decode.sh --nj 1 --cmd "utils/run.pl" exp/mono0a/graph_tgpr data/test_yesno exp/mono0a/decode_test_yesno

steps/decode.sh --nj 1 --cmd utils/run.pl exp/mono0a/graph_tgpr data/test_yesno exp/mono0a/decode_test_yesno
decode.sh: feature type is delta
steps/diagnostic/analyze_lats.sh --cmd utils/run.pl exp/mono0a/graph_tgpr exp/mono0a/decode_test_yesno
steps/diagnostic/analyze_lats.sh: see stats in exp/mono0a/decode_test_yesno/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,1,2) and mean=1.2
steps/diagnostic/analyze_lats.sh: see stats in exp/mono0a/decode_test_yesno/log/analyze_lattice_depth_stats.log
local/score.sh --cmd utils/run.pl data/test_yesno exp/mono0a/graph_tgpr exp/mono0a/decode_test_yesno
local/score.sh: scoring with word insertion penalty=0.0,0.5,1.0



real	0m1.362s
user	0m1.809s
sys	0m1.987s


In [35]:
# Look at the raw decoder output (lattices)
!ls exp/mono0a/decode_test_yesno/

lat.1.gz       wer_11_0.0  wer_13_0.5  wer_15_1.0  wer_7_0.0  wer_9_0.5
log	       wer_11_0.5  wer_13_1.0  wer_16_0.0  wer_7_0.5  wer_9_1.0
num_jobs       wer_11_1.0  wer_14_0.0  wer_16_0.5  wer_7_1.0
scoring_kaldi  wer_12_0.0  wer_14_0.5  wer_16_1.0  wer_8_0.0
wer_10_0.0     wer_12_0.5  wer_14_1.0  wer_17_0.0  wer_8_0.5
wer_10_0.5     wer_12_1.0  wer_15_0.0  wer_17_0.5  wer_8_1.0
wer_10_1.0     wer_13_0.0  wer_15_0.5  wer_17_1.0  wer_9_0.0


## Step 8: Evaluation (Calculating Word Error Rate)

ASR pipeline ရဲ့ နောက်ဆုံးအဆင့်ဖြစ်တဲ့ evaluation လုပ်တဲ့အပိုင်းကို ရောက်လာပါပြီ။  
အသေးစိတ်ဖြည့်စွက်ရှင်းပြရရင်တော့ decoder ကနေထွက်လာတဲ့ output နဲ့ ground-truth text file ကိုနှိုင်းယှဉ်ပြီး Word Error Rate (WER) တွက်ချက်တဲ့အပိုင်းပါ။ Kaldi framework မှာတော့ score.sh ဆိုတဲ့ shell script ကို alignment နဲ့ scoring အလုပ်အတွက် သုံးကြပါတယ်။  

In [36]:
%%bash

# Run the scoring script
local/score.sh --cmd "utils/run.pl" data/test_yesno data/lang_test_tg exp/mono0a/decode_test_yesno

local/score.sh --cmd utils/run.pl data/test_yesno data/lang_test_tg exp/mono0a/decode_test_yesno
local/score.sh: scoring with word insertion penalty=0.0,0.5,1.0


In [37]:
%%bash

# Find the best WER
grep WER exp/mono0a/decode_test_yesno/wer_* | utils/best_wer.sh

%WER 0.00 [ 0 / 232, 0 ins, 0 del, 0 sub ] exp/mono0a/decode_test_yesno/wer_10_0.0


- `WER 0.00` means 0% Word Error Rate! The model recognized everything perfectly.
- `0 / 232` means out of 232 total words in the test set.
- `0 ins, 0 del, 0 sub` means 0 insertions, 0 deletions, and 0 substitutions.

## Summary  

ဒီ ASR tutorial မှာတော့ စာလုံးအရေအတွက်ကလည်း "YES" နဲ့ "NO" နှစ်လုံးတည်းဖြစ်ပြီး grammar အနေနဲ့ကလည်း အရမ်းလွယ်တာမို့ WER က zero ရတာပါ။ လက်တွေ့ တကယ်တမ်း လုပ်ရတဲ့ ASR အလုပ်တွေမှာက WER=0 ဆိုတာက မရှိပါဘူး။  

ဆရာတို့ လုပ်ခဲ့တဲ့ အဆင့်တွေကို ပြန်သုံးသပ်ရရင် အောက်ပါအတိုင်းပါ။  

`Raw Audio ---> Lexicon ---> FSTs ---> MFCCs ---> Model ---> HCLG ---> Decoding ---> WER!`  

တကယ်ကတော့ အဆင့်တိုင်းကို အသေးစိတ်နားလည်ဖို့က အချိန်ပေးလေ့လာရပါလိမ့်မယ်။  


## Appendix

In [40]:
%pwd

'/mnt/disk1/ye/exp/speech/kaldi/egs/yesno_demo/s5'

In [41]:
!tree .

.
├── conf
│   ├── mfcc.conf
│   └── topo_orig.proto
├── data
│   ├── lang
│   │   ├── L_disambig.fst
│   │   ├── L.fst
│   │   ├── oov.int
│   │   ├── oov.txt
│   │   ├── phones
│   │   │   ├── align_lexicon.int
│   │   │   ├── align_lexicon.txt
│   │   │   ├── context_indep.csl
│   │   │   ├── context_indep.int
│   │   │   ├── context_indep.txt
│   │   │   ├── disambig.csl
│   │   │   ├── disambig.int
│   │   │   ├── disambig.txt
│   │   │   ├── extra_questions.int
│   │   │   ├── extra_questions.txt
│   │   │   ├── nonsilence.csl
│   │   │   ├── nonsilence.int
│   │   │   ├── nonsilence.txt
│   │   │   ├── optional_silence.csl
│   │   │   ├── optional_silence.int
│   │   │   ├── optional_silence.txt
│   │   │   ├── roots.int
│   │   │   ├── roots.txt
│   │   │   ├── sets.int
│   │   │   ├── sets.txt
│   │   │   ├── silence.csl
│   │   │   ├── silence.int
│   │   │   ├── silence.txt
│   │   │   ├── wdisambig_phones.int
│   │   │   ├── wdisambig.txt
│   │   │   └── wdisambig_words.int